# Information Extraction
Information Extraction (IE) is the task of automatically extracting structured information from unstructured and/or semi-structured machine-readable documents. This notebook covers key components of IE: Named Entity Recognition (NER), Relationship Extraction, Event Extraction, and Template Filling.


## 1. Named Entity Recognition (NER)
Named Entity Recognition (NER) involves identifying and classifying key elements in text into predefined categories such as person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc.

We will use `spacy` to demonstrate NER.


In [1]:
import spacy

# Load the English NLP model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print('Downloading language model for the spacy POS tagger')
    from spacy.cli import download
    download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

# Sample text
text = "Apple Inc. is looking at buying U.K. startup for $1 billion. Tim Cook, the CEO of Apple, announced this in San Francisco on Tuesday."

# Process the text
doc = nlp(text)

# Extract and print entities
print(f"{'Text':<20} | {'Label':<15} | {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<20} | {ent.label_:<15} | {spacy.explain(ent.label_)}")


Text                 | Label           | Description
------------------------------------------------------------
Apple Inc.           | ORG             | Companies, agencies, institutions, etc.
U.K.                 | GPE             | Countries, cities, states
$1 billion           | MONEY           | Monetary values, including unit
Tim Cook             | PERSON          | People, including fictional
Apple                | ORG             | Companies, agencies, institutions, etc.
San Francisco        | GPE             | Countries, cities, states
Tuesday              | DATE            | Absolute or relative dates or periods


## 2. Relationship Extraction
Relationship Extraction is the task of identifying semantic relationships between entities in text. For example, identifying the "founder of" relationship between "Steve Jobs" and "Apple".

We can use dependency parsing to extract simple subject-verb-object relationships.


In [2]:
# Sample text
text = "Bill Gates founded Microsoft. Mark Zuckerberg created Facebook."

# Process the text
doc = nlp(text)

print("Extracted Relationships:")
# Naive approach using dependency parsing
for token in doc:
    # Look for verbs
    if token.pos_ == "VERB":
        subject = None
        obj = None
        # Find subject and object children
        for child in token.children:
            if child.dep_ in ("nsubj", "nsubjpass"):
                subject = child.text
            if child.dep_ in ("dobj", "pobj", "attr"):
                obj = child.text
        
        if subject and obj:
            print(f"[{subject}] --({token.lemma_})--> [{obj}]")


Extracted Relationships:
[Gates] --(found)--> [Microsoft]
[Zuckerberg] --(create)--> [Facebook]


## 3. Event Extraction
Event Extraction aims to identify events in free text and derive detailed information about them, such as who did what to whom, when, where, and why. 

An event typically consists of an **event trigger** (the word that expresses the event occurrence) and **event arguments** (the participants, time, place, etc.).


In [9]:
# Sample text
text = "Microsoft acquired LinkedIn on January 15th for 16 billion dollars in California."

# Process the text
doc = nlp(text)

# Let's extract an "Acquisition" event
trigger_verb = "acquire"

event = {
    "Type": "Acquisition",
    "Trigger": None,
    "Acquirer": None,
    "AcquiredEntity": None,
    "Date": None,
    "Location": None,
    "Value": None
}

for token in doc:
    if token.lemma_ == trigger_verb:
        event["Trigger"] = token.text
        for child in token.children:
            if child.dep_ == "nsubj":
                event["Acquirer"] = child.text
            if child.dep_ == "dobj":
                event["AcquiredEntity"] = child.text
            if child.dep_ == "prep": # looking for 'on' (time), 'in' (location), 'for' (value)
                for grandchild in child.children:
                    print(f"Processing grandchild: {grandchild}")
                    if child.text == "on" and grandchild.ent_type_ == "DATE":
                        event["Date"] = grandchild.text + " " + " ".join([t.text for t in grandchild.rights])
                    if child.text == "in" and grandchild.ent_type_ == "GPE":
                        event["Location"] = grandchild.text + " " + " ".join([t.text for t in grandchild.rights])
                    if child.text == "for" and grandchild.ent_type_ == "MONEY":
                        event["Value"] = grandchild.text + " " + " ".join([t.text for t in grandchild.rights])

print("Extracted Event:")
for key, value in event.items():
    print(f"{key}: {value}")


Processing grandchild: 15th
Processing grandchild: dollars
Processing grandchild: California
Extracted Event:
Type: Acquisition
Trigger: acquired
Acquirer: Microsoft
AcquiredEntity: LinkedIn
Date: 15th 
Location: California 
Value: dollars 


In [10]:
# Extract and print entities
print(f"{'Text':<20} | {'Label':<15} | {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<20} | {ent.label_:<15} | {spacy.explain(ent.label_)}")

Text                 | Label           | Description
------------------------------------------------------------
Microsoft            | ORG             | Companies, agencies, institutions, etc.
LinkedIn             | ORG             | Companies, agencies, institutions, etc.
January 15th         | DATE            | Absolute or relative dates or periods
16 billion dollars   | MONEY           | Monetary values, including unit
California           | GPE             | Countries, cities, states


## 4. Template Filling
Template Filling is a task where the goal is to map information extracted from text into a predefined set of slots or a template. It is often used in domain-specific applications, like parsing resumes or extracting flight information.

We can use Regular Expressions (Regex) as a simple rule-based approach to fill a template.


In [4]:
import re

# Sample text
text = "I would like to book a flight from Boston to San Francisco on 2023-10-25."

# Define a regex pattern to capture the origin, destination, and date
# Assume format: from [Origin] to [Destination] on [Date]
pattern = r"from\s+([A-Z][A-Za-z\s]+?)\s+to\s+([A-Z][A-Za-z\s]+?)\s+on\s+(\d{4}-\d{2}-\d{2})"

# Define the template
template = {
    "Origin": None,
    "Destination": None,
    "Date": None
}

# Apply regex
match = re.search(pattern, text)

if match:
    template["Origin"] = match.group(1).strip()
    template["Destination"] = match.group(2).strip()
    template["Date"] = match.group(3).strip()
    
print("Filled Template:")
for key, value in template.items():
    print(f"{key}: {value}")
else:
    if not match:
        print("Could not extract information to fill the template.")


Filled Template:
Origin: Boston
Destination: San Francisco
Date: 2023-10-25


## Conclusion
In this notebook, we demonstrated the four main tasks of Information Extraction: Named Entity Recognition, Relationship Extraction, Event Extraction, and Template Filling. State-of-the-art IE often utilizes advanced deep learning and transformer models, but rule-based and dependency-based approaches provide a strong foundation and intuitive understanding of the process.
